# Chapter `2.2.2` - Agent State Management

### Setup and configuration

#### Importing the necessary libraries

In [2]:
# Base utils.
from os import getenv
from dotenv import load_dotenv
import warnings

from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

# State management
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langchain.agents import AgentState

from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

#### Environment settings

In [3]:
load_dotenv()

try:
    OLLAMA_MODEL = getenv("OLLAMA_MODEL", "")
    if not len(OLLAMA_MODEL):
        raise EnvironmentError("Missing Ollama model configuration in environment.")
except EnvironmentError as ee:
    print(f"ERROR: {ee}")

#### Supressing warnings

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph / Pydantic noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")
warnings.filterwarnings("ignore", module="pydantic")

True

### Mutable state management

#### Providing a _mutable_ state

In [6]:
class ColourState(AgentState):
    favourite_colour: str = "blue"  # type: ignore
    least_favourite_colour: str = "red"  # type: ignore

#### Tools for allowing the agent to _access_ and _modify_ runtime state

In [7]:
@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user."""
    return runtime.state.get("favourite_colour", "blue")


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user."""
    return runtime.state.get("least_favourite_colour", "blue")


@tool
def update_favourite_color(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they have revealed it to you."""
    return Command[tuple[()]](
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "I have successfully updated your favourite colour.",
                    tool_call_id=runtime.tool_call_id,
                ),
            ],
        }
    )

#### Agent initialization

In [8]:
agent_updt = create_agent(
    model=OLLAMA_MODEL,
    state_schema=ColourState,
    tools=[get_favourite_colour, get_least_favourite_colour, update_favourite_color],
    checkpointer=InMemorySaver()
)

#### Now, testing across multiple calls

In [9]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

Your favourite colour is blue.

In [11]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="Change my favourite colour to gray.")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

OK. I've updated your favourite colour to gray.

In [13]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="What are my favourite and least favourite colours?")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

Your favourite colour is gray and your least favourite colour is blue.